# Week 2 — Hybrid Model (Vicsek Alignment + Anisotropic Noise)

Vicsek alignment for heading consensus, with noise suppressed by the fraction of neighbors in a forward cone. At λ=0 this is pure Vicsek; at λ=1 agents with all neighbors ahead get zero noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import json
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

FIGURES_DIR = 'figures/'
FPS = 25
DT  = 1.0 / FPS

FIELD_POL_MEAN   = 0.8196
FIELD_POL_STD    = 0.0615
FIELD_NND_MEDIAN = 3.893
FIELD_NND_MEAN   = 4.473
FIELD_TA_STD     = 0.276

Lx = 99.44857450176137
Ly = 55.93982315724077

print("Ready.")

In [ ]:
def run_hybrid(n_agents, Lx, Ly, speed, eta_base, lambda_pull, cone_half_angle,
               r_interaction, r_repulsion, dt, n_steps, seed=42):
    """
    Hybrid model: Vicsek alignment + anisotropic noise from forward-cone geometry.
    
    Update rule:
    1. Vicsek alignment: mean heading of neighbors within r_interaction (including self)
    2. Forward fraction: f_forward = (neighbors in forward cone) / (all neighbors)
    3. Anisotropic noise: eta_eff = eta_base * (1 - lambda_pull * f_forward)
    4. new_heading = mean_heading + noise(eta_eff)
    5. Short-range repulsion override
    
    At lambda_pull=0 this is pure Vicsek. At lambda_pull=1, agents with all
    neighbors ahead get zero noise.
    """
    rng = np.random.default_rng(seed)
    
    pos     = rng.uniform([0, 0], [Lx, Ly], size=(n_agents, 2))
    heading = rng.uniform(-np.pi, np.pi, size=n_agents)
    
    pos_history     = np.empty((n_steps + 1, n_agents, 2))
    heading_history = np.empty((n_steps + 1, n_agents))
    pos_history[0]     = pos
    heading_history[0] = heading
    
    for step in range(n_steps):
        # Pairwise displacements with periodic BC
        delta = pos[np.newaxis, :, :] - pos[:, np.newaxis, :]
        delta[:, :, 0] -= Lx * np.round(delta[:, :, 0] / Lx)
        delta[:, :, 1] -= Ly * np.round(delta[:, :, 1] / Ly)
        dist = np.hypot(delta[:, :, 0], delta[:, :, 1])
        
        # --- Vicsek alignment (identical to Asher's implementation) ---
        align_mask = dist < r_interaction
        unit_vecs  = np.exp(1j * heading)
        neighbour_sum = align_mask @ unit_vecs
        mean_heading  = np.angle(neighbour_sum)
        
        # --- Forward fraction for anisotropic noise ---
        bearing_abs = np.arctan2(delta[:, :, 1], delta[:, :, 0])
        rel_bearing = bearing_abs - heading[:, np.newaxis]
        rel_bearing = (rel_bearing + np.pi) % (2 * np.pi) - np.pi
        
        in_range = (dist < r_interaction) & (dist > 0)
        in_cone  = in_range & (np.abs(rel_bearing) < cone_half_angle)
        
        n_in_range = in_range.sum(axis=1)
        n_in_cone  = in_cone.sum(axis=1)
        f_forward  = n_in_cone / np.maximum(n_in_range, 1)
        
        # --- Anisotropic noise ---
        eta_eff = eta_base * (1.0 - lambda_pull * f_forward)
        noise = rng.uniform(-0.5, 0.5, size=n_agents) * eta_eff
        
        new_heading = mean_heading + noise
        
        # --- Short-range repulsion (same as Vicsek) ---
        rep_mask = (dist < r_repulsion) & (dist > 0)
        has_rep  = rep_mask.any(axis=1)
        if has_rep.any():
            rep_dx = -(rep_mask * delta[:, :, 0]).sum(axis=1)
            rep_dy = -(rep_mask * delta[:, :, 1]).sum(axis=1)
            rep_heading = np.arctan2(rep_dy, rep_dx)
            new_heading[has_rep] = rep_heading[has_rep] + noise[has_rep]
        
        heading = new_heading
        
        pos[:, 0] = (pos[:, 0] + speed * dt * np.cos(heading)) % Lx
        pos[:, 1] = (pos[:, 1] + speed * dt * np.sin(heading)) % Ly
        
        pos_history[step + 1]     = pos
        heading_history[step + 1] = heading
    
    return pos_history, heading_history

print("Hybrid model defined.")

## Metric helpers

In [ ]:
def polarization_from_headings(heading_history):
    return np.abs(np.exp(1j * heading_history).mean(axis=1))

def sim_turning_angles(heading_history):
    dtheta = np.diff(heading_history, axis=0)
    return ((dtheta + np.pi) % (2 * np.pi) - np.pi).ravel()

def sim_nnd(pos_history, Lx, Ly, subsample=20):
    nnds = []
    for t in range(0, len(pos_history), subsample):
        pos = pos_history[t]
        delta = pos[np.newaxis, :, :] - pos[:, np.newaxis, :]
        delta[:, :, 0] -= Lx * np.round(delta[:, :, 0] / Lx)
        delta[:, :, 1] -= Ly * np.round(delta[:, :, 1] / Ly)
        dist = np.hypot(delta[:, :, 0], delta[:, :, 1])
        np.fill_diagonal(dist, np.inf)
        nnds.extend(np.min(dist, axis=1))
    return np.array(nnds)

def sim_neighbor_density_map(pos_history, heading_history, Lx, Ly,
                              radius=10.0, nbins=50, subsample=5):
    edges = np.linspace(-radius, radius, nbins + 1)
    hist = np.zeros((nbins, nbins))
    for t in range(0, len(pos_history), subsample):
        pos = pos_history[t]
        theta = heading_history[t]
        n = len(pos)
        delta = pos[np.newaxis, :, :] - pos[:, np.newaxis, :]
        delta[:, :, 0] -= Lx * np.round(delta[:, :, 0] / Lx)
        delta[:, :, 1] -= Ly * np.round(delta[:, :, 1] / Ly)
        dist = np.hypot(delta[:, :, 0], delta[:, :, 1])
        for i in range(n):
            in_range = (dist[i] > 0.1) & (dist[i] < radius)
            if not in_range.any():
                continue
            rel = delta[i, in_range]
            rot_angle = np.pi / 2 - theta[i]
            cos_r, sin_r = np.cos(rot_angle), np.sin(rot_angle)
            rx = rel[:, 0] * cos_r - rel[:, 1] * sin_r
            ry = rel[:, 0] * sin_r + rel[:, 1] * cos_r
            h, _, _ = np.histogram2d(rx, ry, bins=edges)
            hist += h
    return hist, edges

print("Metric functions defined.")

## 2D Sweep: η_base × λ

In [ ]:
SHARED = dict(
    n_agents=150, Lx=Lx, Ly=Ly,
    speed=11.76, r_interaction=7.0, r_repulsion=1.5,
    dt=DT, cone_half_angle=np.pi / 3,  # 60 deg
)

BURN_IN = 500

eta_vals    = np.arange(0.3, 1.6, 0.1)
lambda_vals = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

pol_grid = np.zeros((len(eta_vals), len(lambda_vals)))

print(f"Sweeping {len(eta_vals)} x {len(lambda_vals)} = {len(eta_vals)*len(lambda_vals)} combinations...")
for i, eta in enumerate(eta_vals):
    for j, lam in enumerate(lambda_vals):
        _, hh = run_hybrid(**SHARED, eta_base=eta, lambda_pull=lam, n_steps=1500, seed=42)
        pol = polarization_from_headings(hh)[BURN_IN:]
        pol_grid[i, j] = pol.mean()
    print(f"  eta={eta:.1f}  pol = {['%.3f' % p for p in pol_grid[i]]}")

# Save sweep arrays
np.save('hybrid_eta_lambda_sweep.npy', pol_grid)
np.save('hybrid_eta_vals.npy', eta_vals)
np.save('hybrid_lambda_vals.npy', lambda_vals)
print("\nSaved: hybrid_eta_lambda_sweep.npy, hybrid_eta_vals.npy, hybrid_lambda_vals.npy")

# Find best (eta, lambda) — closest to field polarization
err = np.abs(pol_grid - FIELD_POL_MEAN)
best_i, best_j = np.unravel_index(err.argmin(), err.shape)
best_eta = eta_vals[best_i]
best_lam = lambda_vals[best_j]
print(f"\nBest: eta_base={best_eta:.1f}, lambda={best_lam:.1f}  pol={pol_grid[best_i, best_j]:.3f} (target={FIELD_POL_MEAN:.3f})")

In [ ]:
# Sweep heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pol_grid, origin='lower', aspect='auto',
               extent=[lambda_vals[0]-0.1, lambda_vals[-1]+0.1,
                       eta_vals[0]-0.05, eta_vals[-1]+0.05],
               cmap='RdYlBu_r', vmin=0.4, vmax=1.0)
ax.plot(best_lam, best_eta, 'k*', markersize=15, label=f'Best: η={best_eta:.1f}, λ={best_lam:.1f}')
ax.set(xlabel='λ (noise suppression)', ylabel='η_base (rad)',
       title='Hybrid Model: Polarization over (η_base, λ)')
plt.colorbar(im, ax=ax, label='Steady-state polarization')

# Contour at field target
cs = ax.contour(lambda_vals, eta_vals, pol_grid, levels=[FIELD_POL_MEAN],
                colors='black', linewidths=2, linestyles='--')
ax.clabel(cs, fmt=f'Φ={FIELD_POL_MEAN:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_noise_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: hybrid_noise_sweep.png")

## Final calibrated run + full metrics

In [ ]:
print(f"Running final hybrid simulation: eta_base={best_eta:.1f}, lambda={best_lam:.1f}")
pos_history, heading_history = run_hybrid(
    **SHARED, eta_base=best_eta, lambda_pull=best_lam, n_steps=2000, seed=42
)

pol_sim    = polarization_from_headings(heading_history)
pol_steady = pol_sim[BURN_IN:]
ta_sim     = sim_turning_angles(heading_history[BURN_IN:])

print(f"Polarization: mean={pol_steady.mean():.3f}  std={pol_steady.std():.3f}  (field: {FIELD_POL_MEAN:.3f} ± {FIELD_POL_STD:.3f})")
print(f"Turning angle std: {np.std(ta_sim):.3f}  (field: {FIELD_TA_STD:.3f})")

print("\nComputing NND...")
nnd_sim = sim_nnd(pos_history[BURN_IN:], Lx, Ly)
print(f"NND: mean={nnd_sim.mean():.2f}  median={np.median(nnd_sim):.2f}  (field: {FIELD_NND_MEAN:.2f} / {FIELD_NND_MEDIAN:.2f})")

print("\nComputing neighbor density map...")
hist_sim, edges = sim_neighbor_density_map(
    pos_history[BURN_IN:], heading_history[BURN_IN:], Lx, Ly, radius=10.0, subsample=10
)
print("Done.")

In [ ]:
# --- Neighbor density map (key figure) ---
fig, ax = plt.subplots(figsize=(7, 6))
hist_norm = hist_sim / (hist_sim.sum() + 1e-10)
im = ax.imshow(hist_norm.T, origin='lower',
               extent=[edges[0], edges[-1], edges[0], edges[-1]],
               cmap='hot', aspect='equal')
ax.plot(0, 0, 'w^', markersize=12)
ax.axhline(0, color='white', ls='--', alpha=0.3)
ax.axvline(0, color='white', ls='--', alpha=0.3)
ax.set(xlabel='Left ← → Right (cm)', ylabel='Behind ← → Ahead (cm)',
       title=f'Hybrid Model — Neighbor density\n(η={best_eta:.1f}, λ={best_lam:.1f})')
plt.colorbar(im, ax=ax, label='Relative density')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_metric_3.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: hybrid_metric_3.png")

# --- Summary table ---
print("\n" + "=" * 65)
print(f"{'Metric':<35} {'Field':>10} {'Vicsek':>10} {'Hybrid':>10}")
print("=" * 65)
print(f"{'Polarization (mean)':<35} {FIELD_POL_MEAN:>10.3f} {'0.824':>10} {pol_steady.mean():>10.3f}")
print(f"{'Polarization (std)':<35} {FIELD_POL_STD:>10.3f} {'0.051':>10} {pol_steady.std():>10.3f}")
print(f"{'Turning angle std (rad)':<35} {FIELD_TA_STD:>10.3f} {'0.595':>10} {np.std(ta_sim):>10.3f}")
print(f"{'NND median (cm)':<35} {FIELD_NND_MEDIAN:>10.2f} {'2.61':>10} {np.median(nnd_sim):>10.2f}")
print(f"{'NND mean (cm)':<35} {FIELD_NND_MEAN:>10.2f} {'3.02':>10} {nnd_sim.mean():>10.2f}")
print("=" * 65)
print(f"\nHybrid params: η_base={best_eta:.1f}, λ={best_lam:.1f}, cone={np.degrees(SHARED['cone_half_angle']):.0f}°")
print(f"Vicsek params: η=1.3 (pure alignment, no geometry)")
print(f"Key comparison: Vicsek needed η=1.3 to match pol; hybrid needs η_base={best_eta:.1f}")

## Export results

In [ ]:
import os

results = {
    "hybrid": {
        "n_agents": SHARED['n_agents'],
        "Lx": SHARED['Lx'],
        "Ly": SHARED['Ly'],
        "speed": SHARED['speed'],
        "eta_base": float(best_eta),
        "lambda_pull": float(best_lam),
        "cone_half_angle": float(SHARED['cone_half_angle']),
        "eta_empirical": 0.26,
        "eta_calibrated": float(best_eta),
        "r_interaction": SHARED['r_interaction'],
        "r_repulsion": SHARED['r_repulsion'],
        "dt": SHARED['dt'],
        "n_steps": 2000,
        "burn_in": BURN_IN,
    },
    "metrics": {
        "polarization_mean": float(pol_steady.mean()),
        "polarization_std": float(pol_steady.std()),
        "turning_angle_std": float(np.std(ta_sim)),
        "nnd_median": float(np.median(nnd_sim)),
        "nnd_mean": float(nnd_sim.mean()),
    },
    "field_targets": {
        "polarization_mean": FIELD_POL_MEAN,
        "polarization_std": FIELD_POL_STD,
        "turning_angle_std": FIELD_TA_STD,
        "nnd_median": FIELD_NND_MEDIAN,
        "nnd_mean": FIELD_NND_MEAN,
    }
}

with open('week2_hybrid_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved: week2_hybrid_results.json")
print(json.dumps(results, indent=2))

# Verify all outputs
print("\n--- Output verification ---")
required = [
    'week2_hybrid_results.json',
    'hybrid_eta_lambda_sweep.npy',
    'hybrid_eta_vals.npy',
    'hybrid_lambda_vals.npy',
    'figures/hybrid_metric_3.png',
    'figures/hybrid_noise_sweep.png',
]
for f in required:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = f"OK ({size:,} bytes)" if exists else "MISSING"
    print(f"  {f}: {status}")